# Escala ampliada: dificuldade × tempo até 50% de AC

**Histórias 50–56; apresentação offline.** Este notebook preserva o espaço `05_demonstracao.ipynb` da história 49 e não altera o artigo do TCC.

O eixo X é a faixa de rating do exercício; o eixo Y é o primeiro tempo em que a CDF de AC atinge pelo menos 0,5. Cada cor identifica uma faixa de **oldRating anterior ao contest**. Os estilos distinguem o KM observado e os modelos congelados.

Run All padrão abre os resultados reais agregados incluídos no projeto. Não baixa contests, não retreina e não refaz joins/bootstrap no modo agregado. O número efetivamente adquirido, as perdas e os hashes são exibidos abaixo; o alvo de 400 não é apresentado como realizado.

In [ ]:
import os, sys, urllib.request
from pathlib import Path

def no_network(*args, **kwargs):
    raise RuntimeError("HTTP desabilitado neste notebook; aquisição é um comando externo explícito")
urllib.request.urlopen = no_network

cwd = Path.cwd().resolve()
STUDY = next((p for p in [cwd, cwd / "study", *cwd.parents]
              if (p / "lorem_study/scale_session.py").is_file()), None)
if STUDY is None:
    raise RuntimeError("Abra o notebook a partir do repositório/study")
sys.path.insert(0, str(STUDY))
os.environ.setdefault("MPLCONFIGDIR", str(STUDY / "work/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(STUDY / "work/cache"))
from lorem_study.scale_session import ScaleSession
from lorem_study.scale_view import plot_curves, display_interactive
from lorem_study.notebook_views import table, notice
import matplotlib.pyplot as plt


## Fonte, alcance e execução

`aggregate` mostra as tabelas/ICs retidos, com integridade verificada. `full` exige a pasta preparada em `LOREM_SCALE_FULL` e recalcula as análises sobre o SQLite verificado, sem rede ou ajuste de modelo. O snapshot completo com audit é um backup separado e não vem no download leve do código. Consulte `SCALE_T50_README.md` para Linux/Windows e restauração.

O universo desde julho/2020 contém menos de 400 contests compatíveis com a exclusão de especiais/versões fácil–difícil. O relatório registra a quantidade exata. Sem submissão não vira fracasso; tentativa válida sem AC é censurada. O rating do problema é o disponível na coleta, sem garantia de ser histórico.

In [ ]:
MODE = os.environ.get("LOREM_SCALE_MODE", "aggregate")
RESULTS = os.environ.get("LOREM_SCALE_RESULTS") or None
FULL_ROOT = os.environ.get("LOREM_SCALE_FULL") or None
OUTPUT = os.environ.get("LOREM_SCALE_OUTPUT") or None
session = ScaleSession(STUDY, mode=MODE, results=RESULTS, full_root=FULL_ROOT, output=OUTPUT)
if MODE == "full":
    session.reanalyze()
report = session.report
table([session.source])
table([session.manifest["acquisition"]])
notice("**Comparação descritiva.** Nova avaliação preditiva independente não realizada. "
       "As amostras anteriores já foram examinadas; aumentar N não cria teste novo.")


## Gráfico combinado

Mude os parâmetros da célula ou use os controles do painel abaixo. Os controles HTML continuam funcionando **offline, sem kernel**, inclusive no arquivo `dashboard.html`. Os agregados permitem seleção/exibição; os ICs existentes não são recalculados ao filtrar linhas. Alterar o protocolo/grade exige a reanálise completa explícita.

A mediana considera eventos e censuras, com AC antes da censura em empates. Quantil não alcançado é ausente com motivo. **120 minutos é um orçamento operacional, não teto da distribuição.** t50 identificado acima de h continua visível. Buracos não são conectados, suavizados ou preenchidos.

O tempo é uma aproximação sequencial de parede, sem medir início de leitura ou dedicação exclusiva. Ratings iguais no Elo não garantem 50% de AC num prazo.

In [ ]:
BANDS = report["plan"]["initial_user_bands"]
METHODS = ("observed",)
PROBLEM_RANGE = (800, 3600)
BUDGET_MINUTES = 120
fig = plot_curves(report, bands=BANDS, methods=METHODS,
                  problem_range=PROBLEM_RANGE, budget=BUDGET_MINUTES)
plt.show(); plt.close(fig)
display_interactive(report, session.fixed)


## Modelos anteriores sobre a mesma população

AFT, KM global e KM por delta são os artefatos completos de 20/10 inventariados antes da nova aquisição. Seus coeficientes, escolhas e hashes são preservados. O snapshot completo de 18/10 e modelos de outros trabalhos externos não foram fornecidos; a consulta antiga retida 1200/1600 não permite reconstruir curvas arbitrárias.

A comparação principal abaixo usa **suporte comum**. O KM observado é reestimado no mesmo subconjunto usado por todos os modelos, com pesos iguais por tentativa. A CDF de cada modelo é misturada nas covariáveis reais e **só então** se extrai t50. A média das medianas seria outro estimando. Cobertura e denominadores aparecem na tabela.

KM global pode gerar linhas coincidentes, sem deslocamento artificial. KM por delta perde informação de nível absoluto. Extrapolação e fallback global oculto são recusados. Linhas dos modelos não representam IC de ajuste.

In [ ]:
COMPARE_BANDS = [1200, 1600]
fig = plot_curves(report, bands=COMPARE_BANDS,
                  methods=("common_observed", "aft", "km_delta", "km_global"))
plt.show(); plt.close(fig)
comparison = []
for cell in report["rows"]:
    if cell["old_lower"] in COMPARE_BANDS:
        comparison.append({"oldRating": [cell["old_lower"], cell["old_upper"]],
            "problema": [cell["problem_lower"], cell["problem_upper"]],
            "N_total": cell["support"]["attempts"], "N_comum": cell["common_support"]["attempts"],
            "cobertura": cell["common_coverage"], "grupos_comuns": cell["common_support"]["groups"],
            "KM_observado_comum_min": cell["common_observed"]["t50_minutes"],
            "AFT_min": cell["predictions"]["aft"]["t50_minutes"],
            "KM_delta_min": cell["predictions"]["km_delta"]["t50_minutes"],
            "motivo_observado": cell["common_observed"]["reason"]})
table(comparison)
table(report["plan"]["model_inventory"])


## Consulta com ratings fixos

Este é um **cenário pontual** de modelo, separado da mistura observada por faixas. Modifique x0 e os ratings explícitos de problema; os artefatos locais permitem novas consultas offline. O painel HTML contém cenários retidos para os inícios positivos das faixas. Cada modelo usa seu próprio domínio neste cenário; um bin de delta ausente não bloqueia AFT/KM global disponíveis.

D-012 fornece duração operacional: aparece somente como coluna/tabela, sem CDF, t50 ou garantia de 50%. Elo de expectativa de vitória, sozinho, não é modelo de tempo até AC.

In [ ]:
X0 = 1200
PROBLEM_RATINGS = [800, 1200, 1600, 2000, 2400, 2800, 3200]
table(session.fixed_query(X0, PROBLEM_RATINGS),
      ["mode", "old_rating", "problem_rating", "method", "t50_minutes", "duration_minutes", "reason"])


## Incerteza, suporte e estabilidade

O bootstrap usa grupos transitivos conhecidos de problemas compartilhados, com reposição e multiplicidade, seed e 200 réplicas pré-fixadas. Se **qualquer réplica** não identifica t50, o IC principal fica ausente; a réplica não é descartada. Os pisos 50 tentativas/20 eventos/5 grupos filtram a exibição e não garantem precisão.

Os ICs são de amostragem por grupo do KM, com os modelos congelados; não incluem incerteza de ajuste. Usuários podem se repetir entre grupos e aliases podem não ser detectados. As réplicas e os resultados por grupo/leave-one-group-out estão no JSON completo.

A tabela usa a região previamente definida oldRating [1200,1600), problema [1600,1800); não foi escolhida por resultado favorável.

In [ ]:
region = next(c for c in report["rows"] if c["old_lower"] == 1200 and c["problem_lower"] == 1600)
uncertainty_rows = []
for key in ("observed", "common_observed"):
    value = region[key]; u = value["uncertainty"]
    uncertainty_rows.append({"população": key, "t50_min": value["t50_minutes"],
        "IC95_min": u["ci95_minutes"], "réplicas": u["replicates"],
        "réplicas_ausentes": u["missing_replicates"], "fração_ausente": u["unavailable_fraction"],
        "motivo": u["reason"]})
table(uncertainty_rows)
table([{"grupo": g, "t50_grupo_min": t,
        "t50_sem_grupo_min": region["observed"]["uncertainty"]["leave_one_group_out_t50_minutes"][g]}
       for g, t in region["observed"]["uncertainty"]["by_group_t50_minutes"].items()])
table([{"participantes_repetidos_entre_grupos": report["participants_repeated_across_groups"],
        "fora_grade_validas": report["outside_grid_valid"],
        "fora_grade_observadas": report["outside_grid_observed"],
        "avaliação_preditiva_nova": report["new_independent_predictive_evaluation"]}])


## Sensibilidades pré-fixadas

As seis combinações de largura oldRating 200/400 × problema 100/200/400 são exportadas integralmente. A tabela abaixo apresenta todas, sem selecionar a melhor. Última submissão e tempo desde o início do contest são estimandos diferentes, identificados separadamente; não são correções validadas da janela principal. Suporte baixo, censura e composição da amostra continuam limitando a interpretação.

In [ ]:
table([{"largura_oldRating": s["old_width"], "largura_problema": s["problem_width"],
        "células": len(s["rows"]),
        "t50_identificados": sum(r["t50_minutes"] is not None for r in s["rows"])}
       for s in report["grid_sensitivity"]])
table(region["support_sensitivity"])
table([{"estimando": k, **v} for k, v in region["estimand_sensitivity"].items()])
notice("Sobreposição com treino/validação examinados e ligações conhecidas disponível em "
       "report['overlap_with_examined_models']; nenhuma alegação de superioridade ou transporte para Ipsums.")


## Exportar e reproduzir

Os gráficos usam os mesmos pontos da tabela CSV/JSON. O painel permite baixar a seleção e SVG/PNG. A célula abaixo salva os dados agregados, as figuras principais e o HTML interativo em uma pasta de saída separada. Para reconstituir observações, joins e bootstrap, restaure o backup completo e use modo full; para novos dados, rode o coletor explícito. Os comandos e checksums estão em `SCALE_T50_README.md`.

**TESTE MANUAL OPCIONAL - NÃO EXECUTADO.** Testes automatizados, execuções limpas e quantidade real constam da evidência. D-012, time-curve-v1 e o aplicativo Android não são alterados por esta análise.

In [ ]:
EXPORTS = session.export()
print("Exports offline:", EXPORTS)
print("Fonte:", session.source)
